# RAG workshop: Введення та основні поняття

Що буде сьогодні:
Для чого
Термінологія
Крок по кроку 


Що потрібно щоб відтворити код в ноутбуці: 
**Потрібен:** OpenAI API ключ (використовуємо `text-embedding-3-*` + `gpt-*`)


## 00 Introduction

![Fine-tuning vs RAG — навички vs знання](https://raw.githubusercontent.com/NatalyUA/RAG-workshop/main/RAG_intro.png)

### Що таке RAG?

**RAG (Retrieval-Augmented Generation)** — підхід, у якому перед відповіддю LLM спочатку **шукають** релевантні фрагменти в зовнішньому корпусі (ваші PDF, таблиці, нотатки), а потім модель **генерує** текст, спираючись на знайдений контекст. Типова схема: **запит → retrieval → підстановка контексту в промпт → відповідь**.


### Why&When, How, For What

![Mind map — RAG: чому, як, які кейси](https://raw.githubusercontent.com/NatalyUA/RAG-workshop/main/RAG_mindmap.png)

**HOW ланцюг:** 
**chunking** (нарізка) → **embedding** (текст → вектори) → **vector store** (зберігання) → **retrieval** (пошук релевантних фрагментів) → **augmentation** (контекст у промпті) → **generation** (відповідь LLM).

In [ ]:
!pip install openai chromadb python-dotenv -q

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

# 🔑 Ключ з файлу .env у корені проєкту (рядок OPENAI_API_KEY=...)
for _root in [Path.cwd(), *Path.cwd().parents]:
    _env = _root / ".env"
    if _env.is_file():
        load_dotenv(_env)
        break
else:
    load_dotenv()

# Альтернатива для Colab: беремо ключ із Secrets (ліва панель -> key icon)
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata

        _colab_key = userdata.get("OPENAI_API_KEY")
        if _colab_key:
            os.environ["OPENAI_API_KEY"] = _colab_key
    except Exception:
        pass

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "Немає OPENAI_API_KEY. Додайте .env у корінь проєкту, змінну середовища або Colab Secret OPENAI_API_KEY."
    )

client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"  # дешева, швидка, 1536 вимірів
LLM_MODEL = "gpt-4o-mini"  # для генерації відповідей

---
## Етап 1: Embedding + Similarity 

**Ідея:** текст → вектор (список чисел), де схожий зміст = близькі числа.


**Embedding** — це процес перетворення тексту або об'єкта на вектор чисел. Кожне число в векторі представляє якусь семантичну властивість тексту. Тексти з подібним значенням матимуть близькі вектори.

**Моделі Embedding:**
- **OpenAI** (`text-embedding-3-small`, `text-embedding-3-large`) — платні, дорогі, але якісні (1536 або 3072 вимірів)
- **Sentence Transformers** (`all-MiniLM-L6-v2`, `all-mpnet-base-v2`) — безплатні, локальні, швидкі (384-768 вимірів)
- **HuggingFace** (`nomic-embed-text`, `UAE-Large-V1`) — різні розміри, можна запустити локально
- **Cohere Embed** — комерційний API, добрий для міцних текстів
- **Word2Vec, FastText** — старіші методи, але ще використовуються для швидких рішень

**Similarity vs Distance:**
- **Cosine Similarity** — вимірює кут між векторами (теоретично від -1 до 1, але на практиці з текстовими embedding'ами часто 0-1). Близькі вектори → значення близько 1.
- **Cosine Distance** — це обернена схожість (значення від 0 до 2, близькі вектори = 0) — саме це повертає ChromaDB при пошуку. Найрелевантніший результат має найменшу distance.

### Допоміжні функції і їх використання 

In [ ]:
# Векторизація (Embedding): текст → числовий вектор
def embed(text: str) -> np.ndarray:
    """Один рядок тексту → один вектор чисел."""
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(response.data[0].embedding)

# Косинусна схожість (cosine similarity): порівняння векторів
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
# Приклад використання. 
word1 = "їдальня"
word2 = "ресторан"

print(f"Схожість між '{word1}' і '{word2}':", cosine_sim(embed(word1), embed(word2)))

# 📝 Завдання #1: 
# Спробуйте власний запит. Що більше схоже на "їдальня" i "кафе" чи "шашлична"? 
# Чи може "аеропорт" бути близьким до "шашличної"?


In [ ]:

# Знайти слова, що схожі на "queen" серед списку слів: : embedding + similarity. 

phrase_for_compare = "queen"
phrases = [
    "queen",
    "королева",
    "king",
    "король",
    "man",
    "woman",
    "prince",
    "princess",
]

vecs = embed(phrase_for_compare)
print(f"Схожість з {phrase_for_compare}  :\n")

scores = []
# Для кожної фрази рахуємо схожість із запитом і зберігаємо пару (фраза, показник схожості)
for phrase in phrases:
    similarity = cosine_sim(vecs, embed(phrase))
    scores.append((phrase, similarity))
    
for phrase, score in sorted(scores, key=lambda x: -x[1]):
    bar = "█" * int(score * 25)
    print(f"  {score:.3f}  {bar:<25}  {phrase}")

# 📝 Завдання #2:     
# a) Порівняйте всі слова в списку зі словом "king"
# b) Додайте до списку ще кілька слів, наприклад "emperor", "president", "builder", 
#    "artist" або інші на ваш вибір, подивіться на їх показники схожості



--- 
🧐 Над ембеддінгами можливі і інші операції, не тільки порівняння. Класичний приклад: у просторі ембеддингів слів виконується **king − man + woman ≈ queen**, про що можна почитати в цікавій науковій статті **Jay Alammar у «The Illustrated Word2Vec».** 



In [ ]:
# Пошук по словосполученям: embedding + similarity
phrase_to_compare = "dog friendly"

phrases = [
    "pet-friendly",
    "dogs/cats allowed",
    "dog-friendly",
    "з собакою дозволяється",
    "з кішкою дозволяється",
    "проживання з собакою заборонено",
    "проживання з домашніми улюбленцями дозволено",
    "проживання з домашніми улюбленцями заборонено",
    "маленькі тварини дозволені",# non-english
    "можна з домашніми улюбленцями", # non-english
    "сучасний ремонт", # нерелевантне, # non-english
    "тварини заборонені", # non-english, антонім
    "animals forbidden",    # антонім
    "free WiFi included",      # нерелевантне
    "parking available nearby", # нерелевантне
]

vecs = embed(phrase_to_compare)

scores = []
for phrase in phrases:
    similarity = cosine_sim(vecs, embed(phrase))
    scores.append((phrase, similarity))
    
for phrase, score in sorted(scores, key=lambda x: -x[1]):
    bar = "█" * int(score * 25)
    print(f"  {score:.3f}  {bar:<25}  {phrase}")
        

# Завдання #3: 
# Спробуйте замінити пошукову фразу на українську, наприклад "можна з собакою"
# Зверніть увагу як зміниться порядок відсортованого по схожесті списку  



--- 
🧐 
### Важливе зауваження: 
Косинусна схожість ембеддингів — корисний інструмент, але не «магія». Якість залежить від того, що саме ви порівнюєте: один довгий документ і одне коротке речення дають різну «концентрацію» змісту в одному векторі; змішані мови, шум, шаблонний текст або рідкісні формулювання теж зміщують оцінку. Тому іноді важливо свідомо підбирати довжину й роль фрагментів (речення, абзац, чанк), а не сліпо довіряти одному числу.

**Практичний напрям покращення** — комбінувати різні типи збігів: порівняння слово до слова (лексика, BM25 тощо), фраза до фрази (семантика через ембеддинги), і гібрид, де обидва сигнали зважуються разом. Є й складніші підходи (reranking, навчені скорери, крос-енкодери тощо) — вони виходять за рамки цього воркшопу,

---
## 📦 Етап 2: Chunking — оголошення про квартиру 🏠

**Проблема:**
- LLM не може з'їсти великий текст цілком — є ліміт токенів.
- Similarity на довгих текстах розмивається.
- Коротке питання примусить аналізувати багатосторінковий документ.

**Рішення:** нарізати текст на шматки і зберігати їх окремо.


In [ ]:
query = "є паркінг?"

listing = """
Здається простора 2-кімнатна квартира в новому ЖК бізнес-класу на Печерську.
Загальна площа — 72 м², житлова — 45 м². Висота стель 3 м, панорамні вікна.
Квартира повністю мебльована та обладнана технікою: кухонний гарнітур, вбудована техніка,
двоспальне ліжко, великий гардероб, пральна машина, посудомийка.
Є окремий кабінет — зручно для роботи вдома.
Інтернет оптоволоконний, швидкість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Квартира має підземний паркінг, який не входить у вартість оренди —
оплачується окремо, 1500 грн/міс. Місце відеоспостереження, охорона цілодобово.
Будинок — закрита територія, консьєрж, дитячий майданчик у дворі.
Тварини — за домовленістю, невеликі породи OK. Неподалік є зручний парк для прогулянок.
Оренда: 28 000 грн/міс. + депозит 1 місяць. Комунальні послуги — окремо.
"""

listing_no_parking = """
Здається простора 2-кімнатна квартира в новому ЖК бізнес-класу на Печерську.
Загальна площа — 72 м², житлова — 45 м². Висота стель 3 м, панорамні вікна.
Квартира повністю мебльована та обладнана технікою: кухонний гарнітур, вбудована техніка,
двоспальне ліжко, великий гардероб, пральна машина, посудомийка.
Є окремий кабінет — зручно для роботи вдома.
Інтернет оптоволоконний, швидкість 1 Гбіт/с — включено у вартість.
Опалення автономне, лічильники на все — платите лише за спожите.
Місце відеоспостереження, охорона цілодобово. 
Будинок — закрита територія, консьєрж, дитячий майданчик у дворі.
Тварини — за домовленістю, невеликі породи OK. Неподалік є зручний парк для прогулянок.
Оренда: 28 000 грн/міс. + депозит 1 місяць. Комунальні послуги — окремо.
"""

# Порівнюємо ЦІЛІ оголошення (без розбиття на чанки)
q_vec_parking = embed(query)

score_with    = cosine_sim(q_vec_parking, embed(listing.strip()))
score_without = cosine_sim(q_vec_parking, embed(listing_no_parking.strip()))

print("=" * 60)
print(f'Запит: «{query}» — порівнюємо ЦІЛІ оголошення')
print("=" * 60)
print(f"  З паркінгом  (слово є в тексті):  score = {score_with:.3f}")
print(f"  Без паркінгу (слова немає):        score = {score_without:.3f}")
print(f"  Різниця:                           {abs(score_with - score_without):.3f}")
print()


---
### Зверніть увагу: 
Різниця мізерна — один факт 'тоне' в довгому векторі.  Саме тому текст розбивають на чанки. Розбиття може бути просто по довжині, а може бути більш складним з урахуванням структури документу (по роздіалах, сторінках), або семантично - по параграфах, закінчених реченях.  

## Допоміжна функцій розбиття на фрагменти і приклад використання 

**Chunking** — нарізка тексту на менші шматки фіксованої довжини. Наприклад, 256 символів на чанк. Це дозволяє модельованню лучше захопити семантику й уникнути розмивання в дуже довгих текстах.

**Overlapping (оверлаппінг)** — перекриття між чанками. Якщо `chunk_size=256` і `overlap=64`, то чанк 1 містить символи 0–256, чанк 2 містить 192–448 (перекривається на 64 символи). Це важливо, щоб не розрізати важливі фрази на межі чанків.


In [ ]:
def chunker(text: str, chunk_size: int = 256, overlap: int = 64) -> list[str]:
    """Нарізає текст на фрагменти з перекриттям (overlap)."""
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start : start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunker(listing, 128, 32) 
print(f"Нарізано на {len(chunks)} chunks:\n")
for i, c in enumerate(chunks):
    print(f"[{i+1}] {c}\n")

In [ ]:
# Без overlap:
chunks_without_overlap = chunker(listing, 128, 0)  
print(f"Нарізано на {len(chunks_without_overlap)} chunks:\n")
for i, c in enumerate(chunks_without_overlap):
    print(f"[{i+1}] {c}\n")

In [ ]:
# 🚨 Навіщо overlap: приклад з паркінгом

# Обираємо 3 найрелевантніші чанки до запиту "є паркінг?" (розбиття з оверлаппінгом)
scores = []
for chunk in chunks:
    similarity = cosine_sim(q_vec_parking, embed(chunk))
    scores.append((chunk, similarity))

for chunk, score in sorted(scores, key=lambda x: -x[1])[:3]:
    print(f"  {score:.3f}  {chunk}")    

In [ ]:
# 🚨 Навіщо overlap: приклад з паркінгом. Частина 2

# Обираємо 3 найрелевантніші чанки до запиту "є паркінг?" (розбиття БЕЗ оверлаппінгом)
scores = []
for chunk in chunks_without_overlap :
    similarity = cosine_sim(q_vec_parking, embed(chunk))
    scores.append((chunk, similarity))

for chunk, score in sorted(scores, key=lambda x: -x[1])[:3]:
    print(f"  {score:.3f}  {chunk}")   

# 📝 Завдання #4: 
# a) Виведіть не 3 найрелевантніші чанки, а всі, подивіться на показник схожості до втраченого фрагменту 
# b) Що зміниться в коді, якщо треба обрати не три чанкі, в ті, для яких показник схожості більше певного порога, наприклад 0.4?

### Для допитливих: як обирають chunking

**Chunking** — це розбиття великого тексту на менші фрагменти (чанки), щоб їх зручно було індексувати і шукати.

**Overlapping** — це перекриття сусідніх чанків (одна частина тексту потрапляє в обидва). Це зменшує ризик, що важлива фраза розірветься на межі.

**Як обирають довжину чанка?** Не тільки "по символах".
- Практичний старт: 300-800 токенів на чанк, overlap 10-20%
- Якщо текст короткий і щільний (полісі, інструкції): менші чанки
- Якщо текст описовий (статті, блоги): можна більші чанки
- Якщо питання дуже точкові: зменшують chunk_size і трохи збільшують overlap

**Чи тільки по довжині?** Ні, є кілька стратегій:
- Fixed-size: просто ріжемо по довжині (найпростіше)
- By structure: по заголовках, абзацах, пунктах, сторінках
- By sentences: по завершених реченнях
- Semantic chunking: об'єднуємо речення за змістовою близькістю

**Орієнтир якості:**
- Якщо релевантний факт часто "випадає" — chunk завеликий або overlap замалий
- Якщо результатів забагато і шумно — chunk замалий або overlap зависокий

---
## 🗄️ Етап 3: Retrieval: Vector Store — Полісі нерухомості 🏠 

**Retrieval** — це крок пошуку релевантних чанків для запиту: система рахує близькість запиту до векторів у базі і повертає top-k найкращих.

До цього ми тримали вектори в змінних (numpy).  

**Vector Store** = база даних для векторів: зберігає на диску, шукає за мілісекунди.
Vector Store не шукає точний збіг слів, а знаходить найближчі за змістом фрагменти через embedding-вектори.

У RAG це шар retrieval: саме звідси дістаються релевантні чанки для підстановки в промпт.

Використовуємо **ChromaDB** — найпростіший варіант для старту.

### Що треба знати про ChromaDB
- `Client()` — все в оперативній пам'яті: зручно для демо, але зникає при перезапуску
- `PersistentClient(path=...)` — зберігає на диск, дані лишаються між сесіями
- Альтернатива локально: **Qdrant** — швидший і з hybrid search із коробки
- У продакшні: Qdrant Cloud, Pinecone, Weaviate — або pgvector якщо вже є PostgreSQL

- **Важливо про міру схожості**: при hnsw:space: "cosine" ChromaDB повертає cosine distance (від 0 до 2), а не схожість. Схожість = 1 - distance. Тому найрелевантніший чанк має найменшу дистанцію і стоїть першим.


### Коротка довідка по операціях ChromaDB

- `create_collection(name)` — створити нову колекцію
- `get_collection(name)` / `get_or_create_collection(name)` — отримати існуючу або створити
- `add(...)` — додати нові записи (нові `id`)
- `upsert(...)` — додати або оновити за `id`
- `get(...)`, `count()`, `query(...)` — подивитися вміст, кількість, пошук
- `delete(ids=[...])` — видалити окремі записи
- `delete_collection(name)` — видалити всю колекцію

In [ ]:
import chromadb

chroma = chromadb.Client()  # in-memory для демо

# Cтворюємо колекцію. Увага - один раз створюємо колекцію, далі додаємо до неї дані, якщо знов виконати цю команду, то буде повідомлення про помилку:
# InternalError: Collection [realty_policies] already exists 

collection = chroma.create_collection("realty_policies", metadata={"hnsw:space": "cosine"})

In [ ]:


# Полісі нерухомості — серйозні (і не дуже)
policies = [
    """Тварини в оренді.
Кіт — завжди так. Пес до 10 кг — з письмовою згодою сусідів. Пес понад 10 кг — лише якщо вміє
тихо дивитися серіали і не гавкає на доставку. Папуга — заборонено, якщо знає матюки.""",

    """Шум і вечірки.
Вечірки дозволені до 23:00. Після 23:00 — виключно тиха дискотека у навушниках.
Перфоратор та дриль — з 10:00 до 18:00 у будні. Хто порушує — слухає перфоратор від сусідів о 7:00.""",

    """Депозит і повернення.
Депозит становить 1 місяць оренди і повертається протягом 30 днів після виїзду.
Не повертається, якщо залишено більше 3 дір від картин або кіт погриз кут дивана.""",

    """Ремонт і зміни.
Будь-який ремонт потребує письмової згоди орендодавця. Стіни можна фарбувати лише в нейтральні кольори.
Яскравий помаранчевий — заборонений, навіть якщо це "для натхнення".""",

    """Перевірка квартири орендодавцем.
Орендодавець попереджає про візит за 24 години. "Просто проходив повз" — не є поважною причиною.
Перевірка раз на місяць, без фотографування особистих речей орендаря.""",

    """Купівля квартири — документи.
Для угоди потрібно: паспорт, ідентифікаційний код і довідка про доходи.
А також — фото вашого обличчя, коли ви дізналися реальну ціну. Нотаріус зберігає всі копії.""",

    """Купівля квартири — строки.
Угода підписується у нотаріуса. Середній строк оформлення — 2–4 тижні.
Якщо банк, іпотека і забудовник не сперечаються — може бути і швидше. Так буває рідко.""",

    """Паркінг.
Паркінг бронюється за принципом: першим прийшов — першим зайняв.
Велосипеди паркувати у спеціальному місці, не в ліфті — навіть якщо він дуже дорогий.""",
]

# Chunking: нарізаємо кожен полісі на шматки
all_chunks = []
for doc in policies:
    chunks = chunker(doc, chunk_size=180, overlap=20)
    all_chunks.extend(chunks)

print(f"Документів: {len(policies)}, чанків після нарізки: {len(all_chunks)}\n")

# Embedding і додавання до ChromaDB
for i, chunk in enumerate(all_chunks):
    vec = embed(chunk).tolist()
    collection.add(ids=[str(i)], documents=[chunk], embeddings=[vec])

print(f"✅ Додано {collection.count()} чанків у ChromaDB")


In [ ]:
query = "до котрої години можна шуміти?"
q_vec = embed(query).tolist()
res = collection.query(query_embeddings=[q_vec], n_results=5)

# ChromaDB вже повертає відсортованими: нагадуємо що найрелевантніший чанк має найменшу дистанцію і стоїть першим
results = zip(res["distances"][0], res["documents"][0])
for dist, doc in results:
    print(f"[{dist:.3f}] {doc}")
    print()
    

---
## 🧠 Етап 4: Augmentation + Generation — Полісі нерухомості

**Augmentation** — це етап, коли знайдені чанки додаються до запиту як контекст (підсилюють запит фактами з бази).

**Generation** — це етап, коли LLM генерує фінальну відповідь, спираючись на запит + доданий контекст.

Ще раз **RAG**:
1. Знайти релевантні chunks (retrieval)
2. **Вставити їх у промпт** як контекст (augmentation)
3. LLM відповідає **тільки на основі цього контексту** (generation)

In [ ]:
# Шаблон промпту — зберігається окремо, підставляємо пізніше
PROMPT_TEMPLATE = """Ти асистент агентства нерухомості. Відповідай тільки на основі наданого контексту.
Якщо відповіді немає в контексті — скажи, що не знаєш.

Контекст:
{context}

Питання: {question}
Відповідь:"""

# Крок 1: запит користувача
question = "Чи можна тримати собаку у квартирі?"

# Крок 2: шукаємо релевантні чанки у векторній базі
q_vec = embed(question).tolist()
res = collection.query(query_embeddings=[q_vec], n_results=3)
chunks = res["documents"][0]

# Крок 3: формуємо контекст із знайдених чанків
context = "\n---\n".join(chunks)

# Крок 4: підставляємо змінні в шаблон
prompt = PROMPT_TEMPLATE.format(context=context, question=question)

print("📋 Промпт який піде в LLM:")
print(prompt)
print()

# Крок 5: викликаємо LLM
response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": prompt}],
)

# Крок 6: виводимо відповідь
print("🤖", response.choices[0].message.content)


In [ ]:
# Завдання #6. Спробуйте ще кілька питань
#
#   "Коли повертають депозит?",
#   "До котрої години можна робити ремонт?",
#   "Які документи потрібні для купівлі?",


In [ ]:
# 🚨 Питання поза базою — модель не вигадує
question = "Яка мінімальна зарплата для отримання іпотеки?"

q_vec = embed(question).tolist()
res = collection.query(query_embeddings=[q_vec], n_results=3)
context = "\n---\n".join(res["documents"][0])

prompt = PROMPT_TEMPLATE.format(context=context, question=question)

response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": prompt}],
)

print(f"❓ {question}")
print(f"🤖 {response.choices[0].message.content}")



## RAG end-to-end
---
![RAG end-to-end pipeline](https://raw.githubusercontent.com/NatalyUA/RAG-workshop/main/RAG_end-to-end_pipeline.png)
---

### Що далі — реальний кейс:
- Обробка документів
- Формування векторного індексу (persistence mode)
- Промпт інжінірінг
- Чат бот на Телеграм